# Nicheverse on Vizgen MERFISH

**Platform.** Vizgen MERSCOPE / MERFISH (imaging based, targeted gene panel).

**Dataset (real).** Mouse retina MERFISH (Vizgen 2023 release), bundled with the package at
`examples/data/merfish_retina.h5ad`: **113,385 cells x 368 genes, 4 samples**. Raw integer counts are in `.X`, micron
centroids in `obsm['spatial']`, and `obs['sample_id']` holds the four MERFISH runs.

**Units.** `obsm['spatial']` is in **microns** (the MERFISH `global_x` / `global_y` stage
coordinates).

The bundled AnnData ships segmented counts only (no per-molecule table), so this notebook
trains on segmented expression. If you have the Vizgen `detected_transcripts.csv`
(columns `global_x`, `global_y`, `gene`), you can add a transcript-context input exactly as
in the Xenium notebook by passing `platform="merfish"`. Real runs use about 300 epochs.


In [ ]:
import anndata as ad, numpy as np
PLATFORM = "MERFISH (mouse retina)"
adata = ad.read_h5ad("../../examples/data/merfish_retina.h5ad")
assert "spatial" in adata.obsm and "sample_id" in adata.obs
print(adata)
print("samples:", list(adata.obs["sample_id"].unique()),
      "| n_genes:", adata.n_vars,
      "| spatial units ~microns:", adata.obsm["spatial"].max(0).round(0))


## Configure and train

We build a `ModelConfig` (the architecture) and a `TrainConfig` (the optimization / spatial graph), then call `train_model`. The current library default encoder is `mlp_deep` (a SwiGLU pre-norm residual MLP) with the `vq` quantizer, and the neighborhood graph is `knn_radius` (radius 50 um, k = 20). We keep `batch_size=2048` rather than `'auto'`, because an over-large auto batch shrinks the number of optimizer steps per epoch and starves the codebook-diversity term. We run only a handful of demo epochs here so the notebook finishes in minutes; a production run uses about 300 epochs.

In [ ]:
import os
from nicheverse.models import ModelConfig, HierarchicalVQVAE
from nicheverse.training import train_model, TrainConfig

ckpt = "runs/nb_merfish_demo"
os.makedirs(ckpt, exist_ok=True)

mc = ModelConfig(
    input_dim=int(adata.n_vars),
    cell_embedding_dim=64, cell_num_embeddings=256,
    neighborhood_embedding_dim=256, neighborhood_num_embeddings=32,
    use_cross_attention=True,
    gene_names=tuple(adata.var_names.astype(str)),
    encoder_type="mlp_plr", quantizer_type="vq",   # strong on a diverse cohort (library default is mlp_deep)
)
tc = TrainConfig(
    num_epochs=30,             # demo; production ~300 (30 epochs already fills ~197/256 codes here)
    batch_size=2048,
    learning_rate=3e-4,
    spatial_graph="knn_radius", radius=50.0, k_neighbors=20,
    normalize=True, log1p=True, seed=9,   # default seed
)
model, adata = train_model(adata, ckpt, model_config=mc, train_config=tc, sample_col="sample_id")
print("done ->", ckpt)


## Inspect the learned codebook

`train_model` writes the per-cell code assignment to `hierarchical_cell_indices.npz` (key `indices`). A healthy run spreads cells across many codes; a collapsed run puts almost everything in one code.

In [ ]:

# --- Load the codes the model just assigned to every cell ---
import numpy as np, json, os
idx = np.load(os.path.join(ckpt, "hierarchical_cell_indices.npz"))["indices"].ravel()
n_codes = int(model.config.cell_num_embeddings)
u, counts = np.unique(idx, return_counts=True)
print(f"{PLATFORM}: {len(idx)} cells assigned to {len(u)}/{n_codes} cell codes "
      f"(codebook usage {100*len(u)/n_codes:.0f}%)")


In [ ]:

# --- Code-usage bar chart (how many cells fall in each active code) ---
%matplotlib inline
import matplotlib.pyplot as plt
order = np.argsort(counts)[::-1]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(u)), counts[order], color="#3b6ea5")
ax.set_xlabel("cell code (sorted by usage)")
ax.set_ylabel("n cells")
ax.set_title(f"{PLATFORM}: cell-code usage ({len(u)}/{n_codes} codes active)")
plt.tight_layout()
plt.show()


## Top markers per code

For each used code we z-score its mean expression across codes and list the most enriched panel genes. This is a quick biological sanity check that codes track distinct cell states.

In [ ]:

# --- Per-code top-marker table: mean log1p expression per code, z-scored across codes ---
import pandas as pd, scanpy as sc
work = adata.copy()
sc.pp.normalize_total(work); sc.pp.log1p(work)
X = work.X.toarray() if hasattr(work.X, "toarray") else np.asarray(work.X)
genes = np.asarray(work.var_names)
rows = []
for c in u:                                   # only codes that are actually used
    m = X[idx == c].mean(0)
    rows.append(m)
M = np.vstack(rows)                            # (n_used_codes, n_genes)
Z = (M - M.mean(0)) / (M.std(0) + 1e-8)        # z across codes, per gene
topk = 6
recs = []
for r, c in enumerate(u):
    top = genes[np.argsort(Z[r])[::-1][:topk]]
    recs.append({"cell_code": int(c), "n_cells": int((idx == c).sum()),
                 "top_markers": ", ".join(top)})
marker_tbl = pd.DataFrame(recs).sort_values("n_cells", ascending=False).reset_index(drop=True)
print(f"Top {topk} enriched genes per used cell code (first 15 codes shown):")
marker_tbl.head(15)


## Training runtime

The trainer records wall-clock time, throughput (cells/sec), and peak GPU memory to `training_runtime.json`.

In [ ]:

# --- Training runtime report the trainer wrote (real timing on this GPU run) ---
rt_path = os.path.join(ckpt, "training_runtime.json")
runtime = json.load(open(rt_path))
print(json.dumps(runtime, indent=2))
